### pymcによる簡単な例でのベイズ線形回帰

#### python moduleのinstall 

このscriptはpymcを用いないと実行できません。


In [ ]:
import sys
import pymc as pm
from packaging.version import Version

required_version = Version("6.0.0")
current_version = Version(pm.__version__)

print(f"PyMC version: {pm.__version__}")

if current_version < required_version:
    sys.exit(
        f"Error: PyMC >= {required_version} is required, "
        f"but found {current_version}"
    )

print("PyMC version check passed.")

#### 解析


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.mlab as mlab
%matplotlib inline


以下の関数を用いた観測量をパラメタの誤差を含めてを回帰します。
$$
T = -0.3 X_1 + 0.1 sin(10X_2) + \delta
$$
つまり基底関数は$(X_1, \sin(10X_2))$です。
$T$は
ガウスノイズ$\delta$を含めた観測量です。

具体的に関数を表示させます。

In [ ]:
def make_XT(seed=10):
    np.random.seed(seed)


    # Xの範囲
    Xrange = [-1, 1]

    # 解の構成
    # w
    w = np.array([-0.3, 0.1])

    # number of samples
    N = 30

    # noise ,  std dev of gaussian
    sigma = 0.05


    X1 = np.linspace(Xrange[0], Xrange[1], N)
    X2 = np.sin(10*X1)
    X = np.vstack([X1, X2]).T
    # print(X.shape)
    Y0 = w[0]*X[:, 0]+w[1]*X[:, 1]

    T = np.random.normal(Y0, scale=sigma, size=N)
    Yerr = sigma

    # 表示
    plt.figure()
    plt.plot(X[:, 0], Y0, ".-", label="Y0")
    plt.errorbar(X[:, 0], T, Yerr, fmt=".-", label="T, bar=std dev.")
    plt.legend()
    plt.show()
    
    return X,Y0,T,Yerr

g_X_all, g_Y0_all, g_T_all, g_Yerr_all = make_XT()

# _allは図を書くために用いる。

In [ ]:
import random

def rancom_choice(Xall,Tall, n_sample = 5):
    idxall = list(range(Xall.shape[0]))
    idx = random.sample(idxall, n_sample)

    X = Xall[idx]
    T = Tall[idx]

    print("{} samples".format(n_sample))
    return X, T, idx

g_n_sample=5

g_X, g_T, idx = rancom_choice(g_X_all, g_T_all, g_n_sample)

線形回帰解を表示します。

In [ ]:
from sklearn.linear_model import LinearRegression


def fit_linearRegression(X, y):
    """fit X Y with liear regression

    Args:
        X (np.array): descriptor
        y (np.array): target values

    Returns:
        np.array: the coefficients of the linear model
    """
    # reg = LinearRegression(fit_intercept=False, normalize=True)
    reg = LinearRegression(fit_intercept=False)

    reg.fit(X, y)
    print("coef=", reg.coef_.ravel(), "R2=", reg.score(X, y))
    return reg.coef_.ravel()


linear_coef = fit_linearRegression(g_X, g_T)


In [ ]:
def predict_and_plot(X, T, Xall, Y0, linear_coef):
    """predict and plot 

    Args:
        X (np.array): sampled descriptor
        T (np.array): observed target values
        Xall (np.array): all the descriptor
        Y0 (np.array): true target values
        linear_coef (np.array): [description]
    """    
    plt.figure()
    ypall = np.dot(Xall, linear_coef)
    plt.plot(Xall[:, 0], ypall, "-", label="pred.")
    plt.plot(X[:, 0], T, "o", label="selected")
    plt.plot(Xall[:, 0], Y0, "-", label="Y0")
    plt.legend()
    plt.show()


predict_and_plot(g_X, g_T, g_X_all, g_Y0_all, linear_coef)


 同じ問題をMCMCで行います。

MCMCを用いて
具体的に$P(t|x,w,S^{-1} )$からありそうな$w$の分布を出すことができます。

In [ ]:
import os
# os.environ["MKL_THREADING_LAYER"] = "GNU" # for nimiconda

In [ ]:
#import pymc3 as pm
import pymc as pm

def make_linear_model(X,T):
    X0 = X[:, 0]
    X1 = X[:, 1]
    basic_model = pm.Model()

    std = [.1, .1]

    with basic_model:
        #a0 = pm.Normal('a0', mu=0, sd=1)
        #a1 = pm.Normal('a1', mu=0, sd=1)
        a0 = pm.Normal('a0', mu=linear_coef[0], sd=std[0])
        a1 = pm.Normal('a1', mu=linear_coef[1], sd=std[1])
        sigma = pm.HalfNormal('sigma', sd=1)
        # 関数式をY0と同じにすること
        mu = a0*X0 + a1*X1
        T_exp = pm.Normal('Y_exp', mu=mu, sd=sigma, observed=T)

    return basic_model

g_basic_model = make_linear_model(g_X, g_T)

In [ ]:
#pm.model_to_graphviz(basic_model)

In [ ]:
# 最初は開始に時間がかかります。

g_map_estimate = pm.find_MAP(model=g_basic_model)

g_map_estimate

## 注意
pymcのupdateにより、
repository downloadした時に
ファイル"linear_model.pickle"が
存在する場合エラーがでます。
ファイル"linear_model.pickle"を一度削除してください。


In [ ]:
import pickle
g_filename_trace = "linear_model.pickle"
if not os.path.isfile(g_filename_trace):
    with g_basic_model:
        g_n_pm_sample = 5000
        g_trace = pm.sample(g_n_pm_sample)
        # it runs n_sample x jobs
    # virtualbox上のububutu,4core,minicondaでは約10秒で終了します。
    # 一方、Anacondaではかなり時間がかかりました。
    with open(g_filename_trace,"wb") as _f:
        pickle.dump(g_trace, _f)
else:
    with open(g_filename_trace,"rb") as _f:
        g_trace = pickle.load(_f)

以下のversionで動作します。

- arviz = 1.1.0
- pm = 6.0.0
- arviz = 1.1.0


In [ ]:
#_ = pm.plot_posterior(g_trace)

import pymc as pm
import arviz as az

print(pm.__version__)
print(az.__version__)

import arviz_plots as azp

azp.plot_dist(g_trace, group="posterior")


もし
"ImportError: ArviZ is not installed. In order to use `plot_posterior`:"
というエラーが表示されたら、arvizのインストールがうまくいっていません。


In [ ]:
#pm.forestplot(g_trace)
azp.plot_forest(g_trace)

In [ ]:
#pm.traceplot(g_trace);
azp.plot_trace(g_trace)

In [ ]:
#pm.summary(g_trace).round(3)

az.summary(g_trace).round(3)

In [ ]:
#g_trace.varnames
list(g_trace.posterior.data_vars)

In [ ]:
import random
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

def plot_distribution2D(trace, var):
    """
    posterior sample を2D表示する。

    Args:
        trace : InferenceData
        var : ["x", "y"]
    """

    # chain, draw を flatten
    a0 = trace.posterior[var[0]].values.flatten()
    a1 = trace.posterior[var[1]].values.flatten()

    n = min(200, len(a0))

    # random sample
    idx = np.random.choice(len(a0), size=n, replace=False)

    a0small = a0[idx]
    a1small = a1[idx]

    plt.figure(figsize=(6, 6))

    # scatter
    plt.plot(a0small, a1small, ".")

    # KDE contour
    sns.kdeplot(
        x=a0small,
        y=a1small,
        fill=True
    )

    plt.title(f"{n} samples")
    plt.xlabel(var[0])
    plt.ylabel(var[1])

    plt.show()

plot_distribution2D(g_trace, ["a0", "a1"])

# 本来はmultivariate distributionです。
# sample数が増えると綺麗な分布になります。


In [ ]:
from numpy.random import multivariate_normal
from sklearn.mixture import GaussianMixture
import numpy as np
import matplotlib.pyplot as plt

def plot_curve(trace, var, Xall, Y0, X, Y, Yerr):
    """plot y and predicted y curves from posterior samples"""

    plt.figure()
    print("X.shape", X.shape, Y.shape, Y0.shape, Yerr)

    # posterior samples: (chain, draw) → flatten
    a0 = trace.posterior[var[0]].values.flatten()
    a1 = trace.posterior[var[1]].values.flatten()

    a01 = np.vstack([a0, a1]).T

    # mean and covariance fit
    cls = GaussianMixture(n_components=1)
    cls.fit(a01)

    print("means", cls.means_)
    print("covariances", cls.covariances_)

    # Gaussian posterior approximation からサンプル
    n = 200
    w_rand = multivariate_normal(
        mean=cls.means_[0],
        cov=cls.covariances_[0],
        size=n
    )

    for w_rand1 in w_rand:
        Y_rand = np.dot(Xall, w_rand1)
        plt.plot(Xall[:, 0], Y_rand, "-", color="red", alpha=0.1)

    plt.plot(Xall[:, 0], Y0, label="Y0", color="blue")

    plt.errorbar(
        X[:, 0],
        Y,
        yerr=Yerr,
        fmt="o",
        label="Y0+-sigma",
        color="blue"
    )

    plt.legend()
    plt.show()
    
plot_curve(g_trace, ["a0", "a1"], g_X_all, g_Y0_all, g_X, g_T, g_Yerr_all)


In [ ]:
# pm.autocorrplot(g_trace)

azp.plot_autocorr(g_trace)

In [ ]:
#pm.energyplot(trace)
azp.plot_energy(g_trace)